In [19]:
# 5.1.3 Model 3: XGBoost Regressor
# Trained on X_train.csv (unscaled) since tree-based models don't need feature scaling.

import pandas as pd
import numpy as np
import xgboost as xgb
import joblib
import json
from sklearn.model_selection import GridSearchCV
import sys, os

sys.path.append(os.path.dirname(os.path.abspath('__file__')))
from model_utils import evaluate_model, cross_validate_model

MODELLING_DIR = os.path.join("..", "data", "modelling")
MODEL_DIR = os.path.join("..", "models")
os.makedirs(MODEL_DIR, exist_ok=True)

X_train = pd.read_csv(os.path.join(MODELLING_DIR, "X_train.csv"))
X_test = pd.read_csv(os.path.join(MODELLING_DIR, "X_test.csv"))

# price is already log-transformed in the CSV (see Section 3.6) — read as-is
y_train = pd.read_csv(os.path.join(MODELLING_DIR, "y_train.csv"))["price"]
y_test = pd.read_csv(os.path.join(MODELLING_DIR, "y_test.csv"))["price"]

print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")

X_train: (3004, 51) | X_test: (751, 51)


In [20]:
param_grid = {
    "n_estimators": [200, 400],
    "max_depth": [4, 6],
    "learning_rate": [0.05, 0.1],
    "subsample": [0.8, 1.0],
}

grid_search = GridSearchCV(
    xgb.XGBRegressor(random_state=42, objective="reg:squarederror"),
    param_grid,
    scoring="neg_root_mean_squared_error",
    cv=5,
    n_jobs=-1,
)
grid_search.fit(X_train, y_train)

print("Best params:", grid_search.best_params_)
print(f"Best CV score (log-scale RMSE): {-grid_search.best_score_:.4f}")

# save tuning result for reference/reproducibility
with open(os.path.join(MODEL_DIR, "xgboost_best_params.json"), "w") as f:
    json.dump(grid_search.best_params_, f, indent=2)

Best params: {'learning_rate': 0.1, 'max_depth': 4, 'n_estimators': 400, 'subsample': 0.8}
Best CV score (log-scale RMSE): 0.2516


In [21]:
model = xgb.XGBRegressor(**grid_search.best_params_, random_state=42, objective="reg:squarederror")
model.fit(X_train, y_train)
print("Model trained.")

Model trained.


In [22]:
train_metrics = evaluate_model(model, X_train, y_train, label="Train")
print()
test_metrics = evaluate_model(model, X_test, y_test, label="Test")

# save metrics for report reference
with open(os.path.join(MODEL_DIR, "xgboost_metrics.json"), "w") as f:
    json.dump({"train": train_metrics, "test": test_metrics}, f, indent=2)

Train RMSE:  RM 67,705  (19.3% of median price)
Train MAE:   RM 42,201
Train MAPE:  10.8%
Train R2:    0.9573
Train MSE:   4,583,999,967

Test RMSE:  RM 166,007  (46.1% of median price)
Test MAE:   RM 77,421
Test MAPE:  16.6%
Test R2:    0.7496
Test MSE:   27,558,216,224


In [23]:
cv_results = cross_validate_model(
    xgb.XGBRegressor(**grid_search.best_params_, random_state=42, objective="reg:squarederror"),
    X_train, y_train, n_splits=5,
)

5-fold CV (mean +/- std):
  RMSE:  RM 174,354 +/- 27,923  (49.6% of median price)
  MAE:   RM 78,561 +/- 4,985
  MAPE:  18.8% +/- 0.8%
  R2:    0.7113 +/- 0.0511
  MSE:   31,178,984,454 +/- 10,373,781,454


In [24]:
importance_table = pd.Series(model.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print("Top 15 features by importance:")
print(importance_table.head(15))

Top 15 features by importance:
Has_Security                      0.103926
Has_Gymnasium                     0.100591
PropertyType_Condominium          0.075230
Property Size                     0.059363
PropertyType_Service_Residence    0.056496
State_Penang                      0.055085
PropertyType_Flat                 0.048986
Parking Lot                       0.041677
State_Selangor                    0.040174
Has_Swimming_Pool                 0.034729
State_Perak                       0.027681
State_Melaka                      0.026287
Bathroom                          0.025329
Property_Age_Missing              0.021473
Total_Units_Missing               0.019976
dtype: float32


In [25]:
model_path = os.path.join(MODEL_DIR, "xgboost_model.pkl")
joblib.dump(model, model_path)
print(f"Model saved to {model_path}")

Model saved to ..\models\xgboost_model.pkl


In [26]:
# Demo: reload the saved model and predict on an existing test row
import joblib
model = joblib.load(os.path.join(MODEL_DIR, "xgboost_model.pkl"))

sample = X_test.iloc[[0]]           # first row of X_test as a stand-in example
preds_log = model.predict(sample)
preds_rm = np.exp(preds_log)

print(f"Predicted price: RM {preds_rm[0]:,.0f}")
print(f"Actual price:    RM {np.exp(y_test.iloc[0]):,.0f}")

Predicted price: RM 356,875
Actual price:    RM 390,000


In [27]:
print("y_test head (log-price):", y_test.head())
print("y_test dtype:", y_test.dtype)
print("Expected actual price (RM):", np.exp(y_test.iloc[0]))

y_test head (log-price): 0    12.873902
1    12.100712
2    12.089539
3    12.793859
4    12.206073
Name: price, dtype: float64
y_test dtype: float64
Expected actual price (RM): 389999.99999999924
